In [21]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.metrics import accuracy_score
from MLOps_project.utils.common import read_yaml,createDIr
from MLOps_project import logger
from pathlib import Path
import joblib
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

In [3]:
%pwd

'c:\\Users\\NishantHP\\Desktop\\MLOps project\\research'

In [4]:
os.chdir('../')

In [5]:
%pwd

'c:\\Users\\NishantHP\\Desktop\\MLOps project'

In [8]:
class config_trainer:
    def __init__(self):
        self.config= read_yaml(Path('config/config.yaml'))
        self.param= read_yaml(Path('params.yaml'))
        self.shema= read_yaml(Path('schema.yaml'))

    def create_trainer_dir(self):
        config=self.config.model_trainer

        createDIr([config.root_trainer_dir])
        logger.info('model trainer root dir created')
        
        return config

In [17]:
import pickle


class model_trainer:
    def __init__(self,):
        self.config= read_yaml(Path('config/config.yaml'))
        
    def scaling_encoding(self,X,code:int):# code = 1 for fit_transform and code = 0 for transform
        
        config= self.config.model_trainer

        num_feat=make_column_selector(dtype_exclude=['object','category'])
        cat_feat=make_column_selector(dtype_include=['object','category'])

        num_pipeline=Pipeline([
            ('imputer',SimpleImputer(strategy='median')),
            ('scaler',StandardScaler())])

        cat_pipeline=Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('encoder',OneHotEncoder(handle_unknown='ignore', drop='first'))])

        preprocessor=ColumnTransformer([
            ('num',num_pipeline,num_feat),
            ('cat',cat_pipeline,cat_feat)])
        
        if code == 1:
            X_done=preprocessor.fit_transform(X)
            with open(config.preprocess_loc,'wb') as f:
                pickle.dump(preprocessor,f)
            logger.info('preprocessor has been dumped')
            return X_done

        elif code == 0:
            with open(config.preprocess_loc,'rb') as f:
                preprocessor = pickle.load(f) 
                logger.info('preprocessor has been loaded')
            return preprocessor.transform(X)

In [23]:
class model_training:
    def __init__(self,config):
        self.config= config

    def model_train_start(self):

        config= self.config

        train = pd.read_csv(config.train_dir)
        test =  pd.read_csv(config.test_dir)

        logger.info('train test data has been loaded')
        X_train=train.drop('Churn',axis=1)
        Y_train=train['Churn']

        X_test=test.drop('Churn',axis=1)
        Y_test=test['Churn']
        logger.info('train test split created')
        scale=model_trainer()
        X_train_preprocessed=scale.scaling_encoding(X_train,1)
        X_test_preprocessed=scale.scaling_encoding(X_test,0)
        logger.info('data has been scaled')
        model = LogisticRegression(
            C=1,
            class_weight='balanced',
            penalty='l1',
            solver='liblinear',
            max_iter=1000
        )
        model.fit(X_train_preprocessed,Y_train)
        logger.info('model has been trained')
        joblib.dump(model,os.path.join(config.model_dir))

        ypred= model.predict(X_test_preprocessed)
        score= accuracy_score(Y_test,ypred)
        print(score)
        

In [24]:
try:
    dir_create=config_trainer()
    config= dir_create.create_trainer_dir()
    model_train=model_training(config)
    train=model_train.model_train_start()
except Exception as e:
    raise e

[2026-04-16 17:48:23,430: INFO: common: yaml file<_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-16 17:48:23,433: INFO: common: yaml file<_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-16 17:48:23,438: INFO: common: yaml file<_io.TextIOWrapper name='schema.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-16 17:48:23,440: INFO: common: created directory at artifacts/model_trainer]
[2026-04-16 17:48:23,441: INFO: 116399586: model trainer root dir created]
[2026-04-16 17:48:23,484: INFO: 624833130: train test data has been loaded]
[2026-04-16 17:48:23,489: INFO: 624833130: train test split created]
[2026-04-16 17:48:23,493: INFO: common: yaml file<_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-16 17:48:23,565: INFO: 1991612955: preprocessor has been dumped]
[2026-04-16 17:48:23,582: INFO: 1991612955: preprocessor has

c:\Users\NishantHP\Desktop\MLOps project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\NishantHP\Desktop\MLOps project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\NishantHP\Desktop\MLOps project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


[2026-04-16 17:48:24,390: INFO: 624833130: model has been trained]
0.7579843860894251
